## EN4554 Assignment 2 — CNNs on CIFAR-10 (PyTorch)

In [1]:
import os
import random
from pathlib import Path

import numpy as np
import torch
import torchvision
from torch.utils.data import Dataset

if torch.cuda.is_available():
    device = torch.device("cuda")
elif getattr(torch.backends, "mps", None) is not None and torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print(f"torch {torch.__version__} | torchvision {torchvision.__version__} | device: {device}")

torch 2.5.1+cu121 | torchvision 0.20.1+cu121 | device: cuda


In [3]:
SEED = 42

def set_seed(seed: int = SEED) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def seed_worker(worker_id: int) -> None:
    """Pass as `worker_init_fn` to DataLoader so each worker is seeded too."""
    worker_seed = torch.initial_seed() % 2**32
    np.random.seed(worker_seed)
    random.seed(worker_seed)

set_seed(SEED)

#### Data — download CIFAR-10 and split 45k train / 5k val

Layout under `data/`:

```
data/
├── raw/                    # torchvision download (cifar-10-batches-py)
└── cifar10_split/
    ├── train/data.pt       # 45,000 images
    ├── val/data.pt         #  5,000 images
    └── test/data.pt        # 10,000 images
```

In [ ]:
DATA_ROOT = Path("data")
RAW_DIR   = DATA_ROOT / "raw"
SPLIT_DIR = DATA_ROOT / "cifar10_split"
SPLITS    = ("train", "val", "test")
N_VAL     = 5_000

CLASSES = ("airplane", "automobile", "bird", "cat", "deer",
           "dog", "frog", "horse", "ship", "truck")

def _split_path(name: str) -> Path:
    return SPLIT_DIR / name / "data.pt"

def _to_tensors(ds):
    # ds.data: (N, 32, 32, 3) uint8  ->  (N, 3, 32, 32) uint8
    x = torch.from_numpy(ds.data).permute(0, 3, 1, 2).contiguous()
    y = torch.tensor(ds.targets, dtype=torch.long)
    return x, y

def prepare_cifar10(seed: int = SEED, n_val: int = N_VAL) -> None:
    if all(_split_path(s).exists() for s in SPLITS):
        print(f"Found existing splits in {SPLIT_DIR}/ — skipping download and split.")
        return

    print("Splits not found — downloading CIFAR-10 (skipped if raw files are already present)...")
    RAW_DIR.mkdir(parents=True, exist_ok=True)
    train_full = torchvision.datasets.CIFAR10(RAW_DIR, train=True,  download=True)
    test_set   = torchvision.datasets.CIFAR10(RAW_DIR, train=False, download=True)

    x_full, y_full = _to_tensors(train_full)
    x_test, y_test = _to_tensors(test_set)

    # Fixed-seed random permutation -> same 45k/5k split every time
    g = torch.Generator().manual_seed(seed)
    perm = torch.randperm(len(x_full), generator=g)
    val_idx, train_idx = perm[:n_val], perm[n_val:]

    splits = {
        "train": (x_full[train_idx], y_full[train_idx]),
        "val":   (x_full[val_idx],   y_full[val_idx]),
        "test":  (x_test,            y_test),
    }
    for name, (x, y) in splits.items():
        _split_path(name).parent.mkdir(parents=True, exist_ok=True)
        torch.save({"images": x, "labels": y, "classes": CLASSES}, _split_path(name))
        print(f"  saved {name:5s}: {tuple(x.shape)} -> {_split_path(name)}")

def load_split(name: str):
    d = torch.load(_split_path(name))
    return d["images"], d["labels"]

prepare_cifar10()

x_train, y_train = load_split("train")
x_val,   y_val   = load_split("val")
x_test,  y_test  = load_split("test")

for name, x, y in [("train", x_train, y_train), ("val", x_val, y_val), ("test", x_test, y_test)]:
    counts = torch.bincount(y, minlength=10).tolist()
    print(f"{name:5s}: {tuple(x.shape)}  per-class counts: {counts}")

Splits not found — downloading CIFAR-10 (skipped if raw files are already present)...


  8%|▊         | 13.5M/170M [04:57<59:33, 43.9kB/s]  

## Per-channel normalization with training-set statistics

Mean and std are computed **only on the 45k training split** and then applied to train, val and test.

In [ ]:
STATS_PATH = SPLIT_DIR / "train_stats.pt"

if STATS_PATH.exists():
    stats = torch.load(STATS_PATH)
    print(f"Loaded cached training statistics from {STATS_PATH}")
else:
    x = x_train.float().div_(255.0)                 # (N, 3, 32, 32) in [0, 1]
    stats = {"mean": x.mean(dim=(0, 2, 3)),          # per-channel over N, H, W
             "std":  x.std(dim=(0, 2, 3))}
    torch.save(stats, STATS_PATH)
    del x

CIFAR_MEAN, CIFAR_STD = stats["mean"], stats["std"]
print("train mean (R, G, B):", [round(v, 4) for v in CIFAR_MEAN.tolist()])
print("train std  (R, G, B):", [round(v, 4) for v in CIFAR_STD.tolist()])


class CIFARTensorDataset(Dataset):
    """uint8 images -> float in [0,1] -> optional augmentation -> per-channel normalization."""

    def __init__(self, images, labels, mean=CIFAR_MEAN, std=CIFAR_STD, augment=None):
        self.images, self.labels = images, labels
        self.mean = mean.view(3, 1, 1)
        self.std  = std.view(3, 1, 1)
        self.augment = augment

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, i):
        x = self.images[i].float().div(255.0)
        if self.augment is not None:
            x = self.augment(x)
        x = (x - self.mean) / self.std
        return x, self.labels[i]


train_ds = CIFARTensorDataset(x_train, y_train)
val_ds   = CIFARTensorDataset(x_val,   y_val)
test_ds  = CIFARTensorDataset(x_test,  y_test)

# Sanity check: normalized training data should be ~0 mean, ~1 std per channel
xb = torch.stack([train_ds[i][0] for i in range(2000)])
print("normalized train sample mean:", xb.mean(dim=(0, 2, 3)).numpy().round(3))
print("normalized train sample std: ", xb.std(dim=(0, 2, 3)).numpy().round(3))